# 02 · ETL e Integração SIH + CNES

**Objetivo:** Carregar os dados traduzidos do SIH e CNES, filtrar internações por IAM (CID I21), realizar limpeza e integrar as duas bases em uma base de modelagem unificada.

**Inputs:**
- `data/interim/sih_*_traduzido.csv` — registros de AIH do SIH com colunas `_DESC` de códigos traduzidos
- `data/interim/cnes_*_traduzido.csv` — tabelas do CNES (ST, LT, EQ, SR, HB) com colunas `_DESC` traduzidas
- `data/external/dicionario_SIH.json` — mapeamento de nomes de colunas do SIH
- `data/external/dicionario_CNES_*.json` — mapeamentos de nomes de colunas do CNES

**Outputs gerados:**
- `data/interim/sih_iam.csv` — internações por IAM (base limpa SIH)
- `data/interim/cnes_hospitais.csv` — base mestre de hospitais (CNES consolidado)
- `data/processed/base_modelagem.csv` — base final SIH × CNES pronta para modelagem

## 0. Configuração do Ambiente

In [1]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path
import sys
import gc

In [2]:
# Adiciona a raiz do projeto ao sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/TCC Documentos/TCC


In [3]:
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Caminhos ───────────────────────────────────────────────────────────
# Os CSVs de entrada agora são os arquivos *traduzidos* gerados pelo
# notebook 01_data_collection.ipynb e salvos em data/interim/
INTERIM   = Path(ROOT, 'data', 'interim')    # traduzidos (input) e outputs do ETL
PROCESSED = Path(ROOT, 'data', 'processed')
EXTERNAL  = Path(ROOT, 'data', 'external')

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print("Caminhos configurados:")
print(f"  Entrada: {INTERIM}")
print(f"  Saída  : {PROCESSED}")

Caminhos configurados:
  Entrada: /home/carolina/Documents/TCC Documentos/TCC/data/interim
  Saída  : /home/carolina/Documents/TCC Documentos/TCC/data/processed


## 1. Carregamento e Padronização do SIH

### 1.1 Leitura dos arquivos

Concatenamos **todos** os arquivos CSV disponíveis em `data/input/SIH/`.
O dicionário `dicionario_SIH.json` é usado para renomear as colunas para nomes legíveis.

In [18]:
# Lê os CSVs traduzidos do SIH (padrão: sih_*_traduzido.csv)
arquivos_sih = sorted(INTERIM.glob('sih_*_traduzido.csv'))
print(f"Arquivos SIH traduzidos encontrados: {len(arquivos_sih)}")
for f in arquivos_sih:
    print(f"  {f.name}")

Arquivos SIH traduzidos encontrados: 121
  sih_rdsp1501_traduzido.csv
  sih_rdsp1502_traduzido.csv
  sih_rdsp1503_traduzido.csv
  sih_rdsp1504_traduzido.csv
  sih_rdsp1505_traduzido.csv
  sih_rdsp1506_traduzido.csv
  sih_rdsp1507_traduzido.csv
  sih_rdsp1508_traduzido.csv
  sih_rdsp1509_traduzido.csv
  sih_rdsp1510_traduzido.csv
  sih_rdsp1512_traduzido.csv
  sih_rdsp1601_traduzido.csv
  sih_rdsp1602_traduzido.csv
  sih_rdsp1603_traduzido.csv
  sih_rdsp1604_traduzido.csv
  sih_rdsp1605_traduzido.csv
  sih_rdsp1606_traduzido.csv
  sih_rdsp1607_traduzido.csv
  sih_rdsp1608_traduzido.csv
  sih_rdsp1609_traduzido.csv
  sih_rdsp1610_traduzido.csv
  sih_rdsp1612_traduzido.csv
  sih_rdsp1701_traduzido.csv
  sih_rdsp1702_traduzido.csv
  sih_rdsp1703_traduzido.csv
  sih_rdsp1704_traduzido.csv
  sih_rdsp1705_traduzido.csv
  sih_rdsp1706_traduzido.csv
  sih_rdsp1707_traduzido.csv
  sih_rdsp1708_traduzido.csv
  sih_rdsp1709_traduzido.csv
  sih_rdsp1710_traduzido.csv
  sih_rdsp1712_traduzido.csv
  

In [19]:
# Carrega e renomeia colunas usando o dicionário externo
path_dicionario = Path(ROOT, 'data', 'external', 'dicionario_SIH.json')
with open(path_dicionario, 'r', encoding='utf-8') as f:
    schema = json.load(f)

rename_dict = {col["old_name"]: col["new_name"] for col in schema}

In [20]:
# Concatena todos os meses aplicando o filtro IAM DURANTE a leitura
# — lê em chunks de 50 000 linhas, filtra por CID I21 imediatamente
# — acumula apenas as linhas IAM
CID_PREFIXO = "I21"
CHUNK_SIZE  = 50_000

frames = []
for i, arq in enumerate(arquivos_sih, 1):
    chunks_iam = []
    try:
        reader = pd.read_csv(arq, dtype=str, low_memory=False, chunksize=CHUNK_SIZE)
        for chunk in reader:
            chunk = chunk.rename(columns=rename_dict)
            chunk["diagnostico_principal"]  = chunk["diagnostico_principal"].astype(str)
            chunk["diagnostico_secundario"] = chunk["diagnostico_secundario"].astype(str)
            mask = (
                chunk["diagnostico_principal"].str.startswith(CID_PREFIXO) |
                chunk["diagnostico_secundario"].str.startswith(CID_PREFIXO)
            )
            iam_chunk = chunk[mask]
            if not iam_chunk.empty:
                chunks_iam.append(iam_chunk)
            del chunk, iam_chunk, mask
            gc.collect()
        if chunks_iam:
            frames.append(pd.concat(chunks_iam, ignore_index=True))
        del chunks_iam
        gc.collect()
        print(f"  [{i:>3}/{len(arquivos_sih)}] {arq.name} — OK")
    except Exception as e:
        print(f"  [{i:>3}/{len(arquivos_sih)}] {arq.name} — ERRO: {e}")
df_iam = pd.concat(frames, ignore_index=True)
del frames
gc.collect()
print(f"\nSIH-IAM carregado: {df_iam.shape[0]:,} registros × {df_iam.shape[1]} colunas")
df_iam.head(3)

  [  1/121] sih_rdsp1501_traduzido.csv — OK
  [  2/121] sih_rdsp1502_traduzido.csv — OK
  [  3/121] sih_rdsp1503_traduzido.csv — OK
  [  4/121] sih_rdsp1504_traduzido.csv — OK
  [  5/121] sih_rdsp1505_traduzido.csv — OK
  [  6/121] sih_rdsp1506_traduzido.csv — OK
  [  7/121] sih_rdsp1507_traduzido.csv — OK
  [  8/121] sih_rdsp1508_traduzido.csv — OK
  [  9/121] sih_rdsp1509_traduzido.csv — OK
  [ 10/121] sih_rdsp1510_traduzido.csv — OK
  [ 11/121] sih_rdsp1512_traduzido.csv — OK
  [ 12/121] sih_rdsp1601_traduzido.csv — OK
  [ 13/121] sih_rdsp1602_traduzido.csv — OK
  [ 14/121] sih_rdsp1603_traduzido.csv — OK
  [ 15/121] sih_rdsp1604_traduzido.csv — OK
  [ 16/121] sih_rdsp1605_traduzido.csv — OK
  [ 17/121] sih_rdsp1606_traduzido.csv — OK
  [ 18/121] sih_rdsp1607_traduzido.csv — OK
  [ 19/121] sih_rdsp1608_traduzido.csv — OK
  [ 20/121] sih_rdsp1609_traduzido.csv — OK
  [ 21/121] sih_rdsp1610_traduzido.csv — OK
  [ 22/121] sih_rdsp1612_traduzido.csv — OK
  [ 23/121] sih_rdsp1701_traduzi

,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,TPDISEC1_DESC,TPDISEC2_DESC,TPDISEC3_DESC,TPDISEC4_DESC,TPDISEC5_DESC,TPDISEC6_DESC,TPDISEC7_DESC,TPDISEC8_DESC,TPDISEC9_DESC,FONTE_ORC
0,350000,2015,1,3,57740490000260.0,3514116185327,1,11713110,354100,19571124,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,350000,2015,1,3,46374500011390.0,3515100592289,1,2804050,355030,19520920,...,Preexistente,Preexistente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,350000,2015,1,3,46374500011390.0,3514120847370,1,2832250,355030,19461225,...,Preexistente,Preexistente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# identificação dos tipos das colunas
df_iam.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 381203 entries, 0 to 381202
Columns: 167 entries, municipio_gestor to FONTE_ORC
dtypes: str(167)
memory usage: 485.7 MB


Todas  as colunas foram identificadas como tipo string, o que devera ser ajustado em breve.

In [22]:
print(f"Internações por IAM carregadas: {len(df_iam):,} registros")

Internações por IAM carregadas: 381,203 registros


### 1.2 Seleção de Colunas Relevantes

In [23]:
df_iam.shape

(381203, 167)

#### Tratamento de Valores Ausentes
Campos administrativos do DataSUS frequentemente chegam como strings vazias ou `"0000"`.
Substituímos apenas **colunas de texto** (object), preservando zeros em variáveis
numéricas legítimas (ex.: `indicador_obito=0` = alta; `uti_mes_total=0` = sem UTI).

In [24]:
# Aplica replace somente em colunas de texto para não corromper variáveis numéricas/binárias
str_cols = df_iam.select_dtypes(include='str').columns
df_iam[str_cols] = df_iam[str_cols].replace(["", "0000", "000"], pd.NA)

# Converte colunas numéricas para tipo correto
num_cols = ["idade", "dias_permanencia", "indicador_obito",
            "uti_mes_total", "codigo_idade"]
for col in num_cols:
    if col in df_iam.columns:
        df_iam[col] = pd.to_numeric(df_iam[col], errors='coerce')

        # IDADE INCOERentE

print("Tratamento de nulos concluído.")

Tratamento de nulos concluído.


Ainda temos 114 colunas e nem todas serão nesessárias então iremos aplicar uma limpeza.
#### Remoção de colunas inutilizáveis

In [25]:
pct_nulos = df_iam.isnull().mean() * 100
print(pct_nulos.sort_values(ascending=False))

INSC_PN_DESC          100.00
SEQ_AIH5_DESC         100.00
infeccao_hospitalar   100.00
data_autorizacao      100.00
cid_notificacao       100.00
                       ...  
MES_CMPT_DESC           0.00
tipo_diag_sec_9         0.00
CAR_INT_DESC            0.00
GESTRISCO_DESC          0.00
FINANC_DESC             0.00
Length: 167, dtype: float64


In [26]:
# ── Remoção de colunas inutilizáveis ─────────────────────────────────────────
# Colunas 100% nulas (sem informação alguma)
limite = 70  # %
cols_remover = pct_nulos[pct_nulos > limite].index
df_iam = df_iam.drop(columns=cols_remover)
print(f'Removidas {len(cols_remover)} colunas com mais de {limite}% de nulos')


# Filtro de idade: manter apenas adultos (>= 18 anos)
#    IAM pediátrico é evento raro e biologicamente distinto — excluído do escopo
df_iam['idade'] = pd.to_numeric(df_iam['idade'], errors='coerce')
n_antes = len(df_iam)
df_iam = df_iam[df_iam['idade'] >= 18].copy()
print(f'Registros pediátricos removidos (idade < 18): {n_antes - len(df_iam)}')
print(f'Base SIH-IAM adultos: {len(df_iam)} registros × {df_iam.shape[1]} colunas')

Removidas 57 colunas com mais de 70% de nulos
Registros pediátricos removidos (idade < 18): 612
Base SIH-IAM adultos: 380591 registros × 110 colunas


In [27]:
# Resumo de completude
nulls = pd.DataFrame({
    "qtd_nulos":  df_iam.isnull().sum(),
    "perc_nulos": df_iam.isnull().mean() * 100
}).sort_values("perc_nulos", ascending=False)

nulls[nulls["qtd_nulos"] > 0]

,qtd_nulos,perc_nulos
diagnostico_secundario_1,204107,53.63
TPDISEC1_DESC,204107,53.63
cnpj_mantenedora,197800,51.97
COBRANCA_DESC,101222,26.60
DIAS_PERM_DESC,85765,22.53
cnpj_hospital,60908,16.00
RACA_COR_DESC,41751,10.97
etnia,5219,1.37
IDENT_DESC,3,0.00
NACIONAL_DESC,3,0.00


### 1.3 Salvar Base SIH-IAM Intermediária

In [28]:
path_sih_interim = INTERIM / "sih_iam.csv"
df_iam.to_csv(path_sih_interim, index=False)
print(f"SIH-IAM salvo em: {path_sih_interim}  ({len(df_iam):,} registros)")

SIH-IAM salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/sih_iam.csv  (380,591 registros)


## 2. Carregamento e Padronização do CNES

Carregamos as cinco tabelas do CNES usando dicionários JSON específicos para cada prefixo.

In [4]:
# Lê os CSVs traduzidos do CNES (padrão: cnes_*_traduzido.csv)
arquivos_cnes = sorted(INTERIM.glob('cnes_*_traduzido.csv'))
print(f"Arquivos CNES traduzidos encontrados: {len(arquivos_cnes)}")
for f in arquivos_cnes:
    print(f"  {f.name}")

Arquivos CNES traduzidos encontrados: 540
  cnes_eq_eqsp1501_traduzido.csv
  cnes_eq_eqsp1502_traduzido.csv
  cnes_eq_eqsp1503_traduzido.csv
  cnes_eq_eqsp1504_traduzido.csv
  cnes_eq_eqsp1505_traduzido.csv
  cnes_eq_eqsp1506_traduzido.csv
  cnes_eq_eqsp1507_traduzido.csv
  cnes_eq_eqsp1508_traduzido.csv
  cnes_eq_eqsp1509_traduzido.csv
  cnes_eq_eqsp1510_traduzido.csv
  cnes_eq_eqsp1512_traduzido.csv
  cnes_eq_eqsp1601_traduzido.csv
  cnes_eq_eqsp1602_traduzido.csv
  cnes_eq_eqsp1603_traduzido.csv
  cnes_eq_eqsp1604_traduzido.csv
  cnes_eq_eqsp1605_traduzido.csv
  cnes_eq_eqsp1606_traduzido.csv
  cnes_eq_eqsp1607_traduzido.csv
  cnes_eq_eqsp1608_traduzido.csv
  cnes_eq_eqsp1609_traduzido.csv
  cnes_eq_eqsp1610_traduzido.csv
  cnes_eq_eqsp1612_traduzido.csv
  cnes_eq_eqsp1701_traduzido.csv
  cnes_eq_eqsp1702_traduzido.csv
  cnes_eq_eqsp1703_traduzido.csv
  cnes_eq_eqsp1704_traduzido.csv
  cnes_eq_eqsp1705_traduzido.csv
  cnes_eq_eqsp1706_traduzido.csv
  cnes_eq_eqsp1707_traduzido.csv
 

In [5]:
def load_cnes_custom(tipo, arquivos_cnes):
    tipo = tipo.lower()

    # Localiza o arquivo traduzido com prefixo cnes_{tipo}_
    arq = next(
        (f for f in arquivos_cnes if f.name.lower().startswith(f'cnes_{tipo}_')),
        None
    )

    if arq is None:
        raise FileNotFoundError(
            f"Arquivo traduzido cnes_{tipo}_*_traduzido.csv não encontrado."
            f" Arquivos disponíveis: {[f.name for f in arquivos_cnes]}"
        )

    # Dicionário de renomeação de colunas (old_name → new_name)
    path_dict = Path(ROOT, 'data', 'external', f'dicionario_CNES_{tipo.upper()}.json')
    with open(path_dict, 'r', encoding='utf-8') as f:
        schema = json.load(f)

    rename_dict = {col["old_name"]: col["new_name"] for col in schema}

    df = pd.read_csv(arq, dtype=str, low_memory=False)

    # Renomeia as colunas originais; as colunas _DESC ficam com o nome gerado
    # automaticamente (ex: TP_LEITO_DESC → mantida como está)
    df = df.rename(columns=rename_dict)

    df['arquivo_origem'] = arq.name
    df['tipo_cnes'] = tipo.upper()

    return df

In [6]:
df_st = load_cnes_custom('st', arquivos_cnes)
df_lt = load_cnes_custom('lt', arquivos_cnes)
df_eq = load_cnes_custom('eq', arquivos_cnes)
df_sr = load_cnes_custom('sr', arquivos_cnes)
df_hb = load_cnes_custom('hb', arquivos_cnes)

print(f"\nRegistros carregados por tabela:")
print(f"  ST (Estabelecimentos) : {df_st.shape}")
print(f"  LT (Leitos)           : {df_lt.shape}")
print(f"  EQ (Equipamentos)     : {df_eq.shape}")
print(f"  SR (Serviços)         : {df_sr.shape}")
print(f"  HB (Habilitações)     : {df_hb.shape}")


Registros carregados por tabela:
  ST (Estabelecimentos) : (66140, 255)
  LT (Leitos)           : (7667, 41)
  EQ (Equipamentos)     : (164782, 43)
  SR (Serviços)         : (104764, 46)
  HB (Habilitações)     : (4494, 49)


### 2.1 pivot Tabelas CNES 

#### Remoção de Duplicidades (ST)

In [7]:
# Garante uma linha por hospital — mantém o registro mais recente
n_antes = len(df_st)
df_st = df_st.drop_duplicates(subset=['codigo_cnes'], keep='last')
print(f"ST: {n_antes} → {len(df_st)} registros únicos (removidas {n_antes - len(df_st):,} duplicatas)")

ST: 66140 → 66140 registros únicos (removidas 0 duplicatas)


#### Agregação de Leitos (LT)

In [8]:
# Pivot de leitos por tipo
df_lt['quantidade_leitos_existentes'] = pd.to_numeric(
    df_lt['quantidade_leitos_existentes'],
    errors='coerce'
).fillna(0)

df_lt_pivot = df_lt.pivot_table(
    index='codigo_cnes',
    columns='tipo_leito',
    values='quantidade_leitos_existentes',
    aggfunc='sum',
    fill_value=0
)

df_lt_pivot.columns = [
    f'leitos_{col}'
    for col in df_lt_pivot.columns
]

df_lt_pivot = df_lt_pivot.reset_index()

#### Agregação de Equipamentos (EQ)

In [9]:
# Pivot de equipamentos
df_eq['quantidade_em_uso'] = pd.to_numeric(
    df_eq['quantidade_em_uso'],
    errors='coerce'
).fillna(0)

df_eq_pivot = df_eq.pivot_table(
    index='codigo_cnes',
    columns='codigo_equipamento',
    values='quantidade_em_uso',
    aggfunc='sum',
    fill_value=0
)

df_eq_pivot.columns = [
    f'equip_{col}'
    for col in df_eq_pivot.columns
]

df_eq_pivot = df_eq_pivot.reset_index()

#### Serviços Especializados (SR) e Habilitações (HB)

In [10]:
# Serviços especializados
df_sr['flag_servico'] = 1

df_sr_pivot = df_sr.pivot_table(
    index='codigo_cnes',
    columns='codigo_servico_especializado',
    values='flag_servico',
    aggfunc='max',
    fill_value=0
)

df_sr_pivot.columns = [
    f'servico_{col}'
    for col in df_sr_pivot.columns
]

df_sr_pivot = df_sr_pivot.reset_index()

In [11]:
# Habilitações
df_hb['flag_habilitacao'] = 1

df_hb_pivot = df_hb.pivot_table(
    index='codigo_cnes',
    columns='codigo_habilitacao',
    values='flag_habilitacao',
    aggfunc='max',
    fill_value=0
)

df_hb_pivot.columns = [
    f'habilitacao_{col}'
    for col in df_hb_pivot.columns
]

df_hb_pivot = df_hb_pivot.reset_index()

In [12]:
print(df_lt_pivot.shape)
print(df_eq_pivot.shape)
print(df_sr_pivot.shape)
print(df_hb_pivot.shape)

(1173, 8)
(35879, 82)
(28645, 66)
(1644, 147)


## 3 Salvar Base Mestre de Hospitais (Interim CNES)

In [29]:
# ── Merge das tabelas CNES em torno da ST (uma linha por hospital) ───────────
#
# ST  : tabela mestre — um registro por hospital (já deduplicada acima)
# LT  : leitos por tipo de leito  → pivotado em df_lt_pivot
# EQ  : equipamentos por código   → pivotado em df_eq_pivot
# SR  : serviços especializados   → pivotado em df_sr_pivot
# HB  : habilitações              → pivotado em df_hb_pivot
#
# Usamos left join para manter todos os hospitais da ST,
# mesmo que não possuam leitos, equipamentos, etc. registrados.

df_cnes_final = df_st.copy()

for df_pivot, nome in [
    (df_lt_pivot, 'LT — Leitos'),
    (df_eq_pivot, 'EQ — Equipamentos'),
    (df_sr_pivot, 'SR — Serviços Especializados'),
    (df_hb_pivot, 'HB — Habilitações'),
]:
    # Garante que a chave está no mesmo formato (string sem espaços)
    df_pivot['codigo_cnes'] = df_pivot['codigo_cnes'].astype(str).str.strip()
    df_cnes_final = pd.merge(df_cnes_final, df_pivot, on='codigo_cnes', how='left')
    print(f'  Após merge {nome}: {df_cnes_final.shape}')

# Preenche NaN nas colunas de contagem/flag com 0
# (hospitais sem leitos/equipamentos/serviços registrados ficam como 0, não NaN)
cols_pivot = (
    list(df_lt_pivot.columns.drop('codigo_cnes')) +
    list(df_eq_pivot.columns.drop('codigo_cnes')) +
    list(df_sr_pivot.columns.drop('codigo_cnes')) +
    list(df_hb_pivot.columns.drop('codigo_cnes'))
)
df_cnes_final[cols_pivot] = df_cnes_final[cols_pivot].fillna(0)

# Remove colunas auxiliares que vieram das tabelas secundárias e são redundantes
# (arquivo_origem e tipo_cnes existem em cada pivot — causam sufixos _x/_y)
cols_remover_aux = [c for c in df_cnes_final.columns if c.endswith(('_x', '_y'))]
if cols_remover_aux:
    df_cnes_final = df_cnes_final.drop(columns=cols_remover_aux)
    print(f'  Colunas auxiliares duplicadas removidas: {cols_remover_aux}')

print(f'\nBase CNES final: {df_cnes_final.shape[0]:,} hospitais × {df_cnes_final.shape[1]} colunas')


  Após merge LT — Leitos: (66140, 262)
  Após merge EQ — Equipamentos: (66140, 343)
  Após merge SR — Serviços Especializados: (66140, 408)
  Após merge HB — Habilitações: (66140, 554)

Base CNES final: 66,140 hospitais × 554 colunas


### 3.1 Limpeza de Colunas com Muitos Nulos

Assim como feito com o SIH, removemos colunas com alto percentual de valores ausentes.
Colunas com `> 70%` de nulos não carregam informação útil para a modelagem e apenas
adicionam ruído e custo computacional.

**Tipos de colunas que tendem a ser removidas:**
- Colunas `_DESC` sem dicionário `.cnv` mapeado (ficam 100% nulas)
- Campos de controle administrativo do CNES nunca preenchidos em SP
- Campos de contrato municipal/estadual que só existem em alguns estados


In [14]:
# ── Inspeção de nulos no CNES após merge ─────────────────────────────────────
pct_nulos_cnes = df_cnes_final.isnull().mean() * 100

# Distribuição por faixa
faixas = {
    '100%'  : (pct_nulos_cnes == 100).sum(),
    '70–99%': ((pct_nulos_cnes >= 70) & (pct_nulos_cnes < 100)).sum(),
    '50–69%': ((pct_nulos_cnes >= 50) & (pct_nulos_cnes < 70)).sum(),
    '1–49%' : ((pct_nulos_cnes > 0)  & (pct_nulos_cnes < 50)).sum(),
    '0%'    : (pct_nulos_cnes == 0).sum(),
}
print(f'Shape antes da limpeza: {df_cnes_final.shape}')
print('\nDistribuição de nulos por coluna:')
for faixa, qtd in faixas.items():
    print(f'  {faixa:8}: {qtd:>4} colunas')

print('\nTop 20 colunas com mais nulos:')
print(pct_nulos_cnes.sort_values(ascending=False).head(20).to_string())


Shape antes da limpeza: (66140, 554)

Distribuição de nulos por coluna:
  100%    :   21 colunas
  70–99%  :   37 colunas
  50–69%  :    0 colunas
  1–49%   :   14 colunas
  0%      :  482 colunas

Top 20 colunas com mais nulos:
data_publicacao_contrato_municipal   100.00
numero_contrato_estadual             100.00
data_publicacao_contrato_estadual    100.00
CLASAVAL_DESC                        100.00
data_avaliacao_pnass                 100.00
COMPETEN_DESC                        100.00
NAT_JUR_DESC                         100.00
GESPRG3E_DESC                        100.00
CODUFMUN_DESC                        100.00
numero_contrato_municipal            100.00
TPGESTAO_DESC                        100.00
RETENCAO_DESC                        100.00
NIV_HIER_DESC                        100.00
TP_PREST_DESC                        100.00
DT_PUBLM_DESC                        100.00
DT_PUBLE_DESC                        100.00
DT_EXPED_DESC                        100.00
ORGEXPED_DESC          

In [15]:
# ── Remoção de colunas com > 70% de nulos ────────────────────────────────────
LIMITE_NULOS = 70  # mesmo critério aplicado ao SIH

cols_remover_cnes = pct_nulos_cnes[pct_nulos_cnes > LIMITE_NULOS].index.tolist()

df_cnes_final = df_cnes_final.drop(columns=cols_remover_cnes)

print(f'Colunas removidas (> {LIMITE_NULOS}% nulos): {len(cols_remover_cnes)}')
print(f'Shape após limpeza: {df_cnes_final.shape}')
print()

# Resumo de completude após limpeza
nulls_cnes = pd.DataFrame({
    'qtd_nulos' : df_cnes_final.isnull().sum(),
    'perc_nulos': df_cnes_final.isnull().mean() * 100
}).sort_values('perc_nulos', ascending=False)

restantes_com_nulos = nulls_cnes[nulls_cnes['qtd_nulos'] > 0]
print(f'Colunas restantes com algum nulo: {len(restantes_com_nulos)}')
if not restantes_com_nulos.empty:
    display(restantes_com_nulos)


Colunas removidas (> 70% nulos): 58
Shape após limpeza: (66140, 496)

Colunas restantes com algum nulo: 14


,qtd_nulos,perc_nulos
SERAP01P_DESC,32952,49.82
RES_BIOL_DESC,31420,47.51
codigo_regiao_saude,26063,39.41
NIV_DEP_DESC,11148,16.86
data_expedicao_alvara,9302,14.06
numero_alvara,8511,12.87
orgao_expedidor_alvara,8433,12.75
avaliado_pnass,7464,11.29
avaliado_acreditacao,6451,9.75
codigo_nivel_hierarquia,6061,9.16


In [16]:
path_cnes_interim = INTERIM / "cnes_hospitais.csv"
df_cnes_final.to_csv(path_cnes_interim, index=False)
print(f"CNES Interim salvo em: {path_cnes_interim}  ({len(df_cnes_final):,} hospitais)")

CNES Interim salvo em: /home/carolina/Documents/TCC Documentos/TCC/data/interim/cnes_hospitais.csv  (66,140 hospitais)


## 4. Fusão Global: SIH-IAM × CNES

Fazemos um *Left Join* das internações por IAM com a base mestre de hospitais.
A chave de junção é `codigo_cnes`, normalizada em ambos os lados para evitar
divergências por zeros à esquerda ou espaços.

In [30]:
# ── Padroniza a chave de merge nos dois DataFrames ──────────────────────────
# No SIH-IAM o campo foi salvo como 'cnes' (nome original do DATASUS);
# no CNES-final a coluna já foi renomeada para 'codigo_cnes' pelo dicionário.
if 'cnes' in df_iam.columns and 'codigo_cnes' not in df_iam.columns:
    df_iam = df_iam.rename(columns={'cnes': 'codigo_cnes'})

df_iam['codigo_cnes']        = df_iam['codigo_cnes'].astype(str).str.strip().str.zfill(7)
df_cnes_final['codigo_cnes'] = df_cnes_final['codigo_cnes'].astype(str).str.strip().str.zfill(7)

# ── Resolve conflitos de nome entre SIH e CNES antes do merge ────────────────
# Colunas que existem nos dois lados (exceto a chave) receberiam sufixo _x/_y.
# Estratégia: renomear as colunas do CNES adicionando prefixo 'cnes_' quando conflitam.
chave = 'codigo_cnes'
cols_sih  = set(df_iam.columns) - {chave}
cols_cnes = set(df_cnes_final.columns) - {chave}
conflitos = cols_sih & cols_cnes

if conflitos:
    print(f'Colunas em conflito renomeadas no CNES (prefixo cnes_): {sorted(conflitos)}')
    df_cnes_final = df_cnes_final.rename(
        columns={c: f'cnes_{c}' for c in conflitos}
    )

# ── Left Join: mantém todos os registros IAM ──────────────────────────────────
df_base_modelagem = pd.merge(df_iam, df_cnes_final, on=chave, how='left')

# ── Log de qualidade do merge ─────────────────────────────────────────────────
# Usa uma coluna que SÓ existe no CNES para medir o match rate
col_check = 'tipo_unidade'  # sempre vem do CNES-ST, nunca do SIH
n_sem_cnes = df_base_modelagem[col_check].isna().sum()

print(f'Total de internações IAM        : {len(df_base_modelagem):,}')
print(f'Sem correspondência no CNES     : {n_sem_cnes:,}  ({n_sem_cnes/len(df_base_modelagem)*100:.1f}%)')
print(f'Com dados hospitalares do CNES  : {len(df_base_modelagem) - n_sem_cnes:,}')
print(f'Número de features da base final: {df_base_modelagem.shape[1]}')


Colunas em conflito renomeadas no CNES (prefixo cnes_): ['cnpj_mantenedora', 'natureza_juridica', 'tipo_gestao']
Total de internações IAM        : 380,591
Sem correspondência no CNES     : 12,310  (3.2%)
Com dados hospitalares do CNES  : 368,281
Número de features da base final: 663


In [31]:
# Exporta base de modelagem final
path_base_final = PROCESSED / "base_modelagem.csv"
df_base_modelagem.to_csv(path_base_final, index=False)
print(f"Base de modelagem salva em: {path_base_final}")
df_base_modelagem.sample(3)

Base de modelagem salva em: /home/carolina/Documents/TCC Documentos/TCC/data/processed/base_modelagem.csv


,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,habilitacao_806,habilitacao_807,habilitacao_901,habilitacao_902,habilitacao_903,habilitacao_904,habilitacao_905,habilitacao_906,habilitacao_907,habilitacao_908
201551,350000,2021,8,1,NaN,3521102708508,1,13085712,350950,19510112,...,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
190461,350000,2021,4,1,53725560000170.0,3521101864654,1,2516010,355030,19621128,...,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
81009,350000,2017,12,1,60742616000160.0,3517128376850,1,8472285,355030,19470727,...,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [32]:
df_base_modelagem

,municipio_gestor,ano_competencia,mes_competencia,especialidade_leito,cnpj_hospital,numero_aih,tipo_aih,cep_paciente,municipio_residencia,data_nascimento,...,habilitacao_806,habilitacao_807,habilitacao_901,habilitacao_902,habilitacao_903,habilitacao_904,habilitacao_905,habilitacao_906,habilitacao_907,habilitacao_908
0,350000,2015,1,3,57740490000260.0,3514116185327,1,11713110,354100,19571124,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,350000,2015,1,3,46374500011390.0,3515100592289,1,2804050,355030,19520920,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,350000,2015,1,3,46374500011390.0,3514120847370,1,2832250,355030,19461225,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,350000,2015,1,3,46374500011390.0,3514121521405,1,2853000,355030,19460703,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,350000,2015,1,1,NaN,3514126947892,1,15460000,351980,19471208,...,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
380586,355650,2025,12,3,NaN,3525111826830,1,13221601,355650,19681205,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
380587,355670,2025,12,3,72909179000105.0,3525123495091,1,13285220,355670,19460204,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
380588,355670,2025,12,3,72909179000105.0,3525123494981,1,13280460,355670,19610613,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
380589,355670,2025,12,3,72909179000105.0,3525123495950,1,13285536,355670,19770325,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


---
## Resumo do Pipeline

| Etapa | Input | Output | Registros |
|-------|-------|--------|-----------|
| 1. Filtro IAM (SIH) | `data/input/SIH/*.csv` | `data/interim/sih_iam.csv` | ver acima |
| 2. Consolidação CNES | `data/input/CNES/*.csv` | `data/interim/cnes_hospitais.csv` | ver acima |
| 3. Fusão global | sih_iam + cnes_hospitais | `data/processed/base_modelagem.csv` | ver acima |

> **Próximos passos:** notebook `03_analise_exploratotia_visualizacao.ipynb` — análise exploratória e visualizações.